In [ ]:
from google.colab import drive
drive.mount('/content/drive')         #drive mount

Mounted at /content/drive


In [ ]:
import shutil

# Source path in Drive
drive_path = '/content/drive/MyDrive/Copy of ROI_Features_Local.zip'        #Imported to colab environment

# Destination path in Colab local
local_path = '/content/ROI_Features_Local.zip'

# Copy the file
shutil.copy(drive_path, local_path)


'/content/ROI_Features_Local.zip'

In [ ]:
import zipfile

zip_path = "/content/ROI_Features_Local.zip"
extract_path = "/content/ROI_Features_Local"            #unzipping

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [ ]:
import os

def find_nonempty_dir(base_dir):
    for root, dirs, files in os.walk(base_dir):
        if len(files) > 0:
            return root
    return None

target_path = find_nonempty_dir("/content/ROI_Features_Local")

if target_path:
    print(f"✅ Found non-empty folder: {target_path}")

    broken_files = []
    empty_files = []
    total_checked = 0

    for root, dirs, files in os.walk(target_path):
        for file in files:                                        #Checking
            total_checked += 1
            file_path = os.path.join(root, file)
            try:
                if os.path.getsize(file_path) == 0:
                    empty_files.append(file_path)
                    continue
                with open(file_path, 'rb') as f:
                    f.read(10)
            except Exception as e:
                broken_files.append((file_path, str(e)))

    print(f"\n📊 File Integrity Check Summary")
    print(f"📁 Checked folder: {target_path}")
    print(f"📄 Total files checked: {total_checked}")
    print(f"🟡 Empty files: {len(empty_files)}")
    print(f"❌ Broken/unreadable files: {len(broken_files)}")

else:
    print("⚠️ No non-empty directory found inside /content/ROI_Features_Local.")



✅ Found non-empty folder: /content/ROI_Features_Local

📊 File Integrity Check Summary
📁 Checked folder: /content/ROI_Features_Local
📄 Total files checked: 28926
🟡 Empty files: 0
❌ Broken/unreadable files: 0


In [ ]:
import json

# Input text file path (replace with actual if needed)
input_txt_path = "/content/hindi-visual-genome-train.txt"
output_json_path = "/content/image_ids_and_captions.json"

output_list = []

with open(input_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")

        if len(parts) >= 7:
            image_id = parts[0].strip()
            english_caption = parts[-2].strip()           #json mapping
            hindi_caption = parts[-1].strip()

            output_list.append({
                "image_id": image_id,
                "english_caption": english_caption,
                "hindi_caption": hindi_caption
            })

# Write to JSON
with open(output_json_path, "w", encoding="utf-8") as f_out:
    json.dump(output_list, f_out, ensure_ascii=False, indent=2)

print(f"✅ JSON saved to: {output_json_path}")


✅ JSON saved to: /content/image_ids_and_captions.json


In [ ]:
import os
import json

# === Paths ===
json_path = "/content/image_ids_and_captions.json"
npy_dir = "/content/ROI_Features_Local"  # Folder with .npy files

# === Load JSON file ===
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# === Build set of available image_ids in .npy files ===
available_ids = set()
for fname in os.listdir(npy_dir):
    if fname.endswith(".npy"):
        image_id = fname.split("_")[0]  # Assumes name like 2322111_xyz.npy     #Missing Check
        available_ids.add(image_id)

# === Find missing image_ids ===
missing_ids = []
for entry in data:
    image_id = str(entry["image_id"])
    if image_id not in available_ids:
        missing_ids.append(image_id)

# === Print result ===
print(f"✅ Total entries in JSON: {len(data)}")
print(f"✅ Found image_ids in npy_dir: {len(available_ids)}")
print(f"❌ Missing .npy files: {len(missing_ids)}")

if missing_ids:
    print("\nExample missing image_ids:")
    print(missing_ids[:20])  # Show first 20


✅ Total entries in JSON: 28930
✅ Found image_ids in npy_dir: 28923
❌ Missing .npy files: 4

Example missing image_ids:
['2322111', '2329621', '2341369', '2379473']


In [ ]:
# === Clean the JSON ===
clean_data = [entry for entry in data if str(entry["image_id"]) in available_ids]

# === Save cleaned JSON ===
cleaned_json_path = "/content/your_dataset_cleaned.json"
with open(cleaned_json_path, "w", encoding="utf-8") as f:
    json.dump(clean_data, f, ensure_ascii=False, indent=2)          #Clean json from missing data

print(f"✅ Cleaned dataset saved to: {cleaned_json_path}")
print(f"✅ Entries retained: {len(clean_data)}")


✅ Cleaned dataset saved to: /content/your_dataset_cleaned.json
✅ Entries retained: 28926


In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset
import numpy as np

class CaptionDataset(Dataset):
    def __init__(self, json_path, feature_dir, tokenizer, max_length=32):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.feature_dir = feature_dir
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Precompute lookup table: {image_id: path_to_npy}
        self.feature_lookup = {}
        for fname in os.listdir(feature_dir):
            if fname.endswith('.npy'):
                base_id = fname.split('_')[0]
                self.feature_lookup[base_id] = os.path.join(feature_dir, fname)

        # Filter out entries with missing features
        self.data = [entry for entry in self.data if entry['image_id'] in self.feature_lookup]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image_id = item['image_id']
        caption = item['caption']  # or 'caption_hindi' etc.

        # Load .npy feature
        feature_path = self.feature_lookup[image_id]
        features = np.load(feature_path)
        features = torch.tensor(features, dtype=torch.float32)

        # Tokenize caption
        tokenized = self.tokenizer(
            caption,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'features': features,            # Shape: [feature_dim]
            'input_ids': tokenized.input_ids.squeeze(0),
            'attention_mask': tokenized.attention_mask.squeeze(0)
        }


In [ ]:
import torch.nn as nn

class CaptioningModel(nn.Module):
    def __init__(self, feature_dim, vocab_size, hidden_dim=512):
        super().__init__()
        self.feature_proj = nn.Linear(feature_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.transformer = nn.Transformer(
            d_model=hidden_dim, nhead=8, num_encoder_layers=3, num_decoder_layers=3
        )
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, input_ids, attention_mask=None):
        # features: [B, feature_dim] → [1, B, hidden_dim]
        encoder_input = self.feature_proj(features).unsqueeze(0)

        # input_ids → embeddings → [T, B, hidden_dim]
        embedded = self.embedding(input_ids).transpose(0, 1)

        output = self.transformer(encoder_input, embedded)
        logits = self.output_layer(output.transpose(0, 1))  # [B, T, vocab_size]
        return logits


In [ ]:
import os
import torch
from torch.utils.data import Dataset
import sentencepiece as spm
import numpy as np
import json

class LazyImageCaptionDataset(Dataset):
    def __init__(self, json_path, npy_dir, sp_model_path, max_len=30, pad_token=0):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.npy_dir = npy_dir
        self.max_len = max_len
        self.pad_token = pad_token

        self.sp = spm.SentencePieceProcessor()
        self.sp.load(sp_model_path)
        self.bos = self.sp.bos_id()
        self.eos = self.sp.eos_id()

        # ✅ Pre-index all .npy file paths by image_id
        self.imageid_to_path = {}
        for f in os.listdir(npy_dir):
            if f.endswith(".npy"):
                image_id = f.split("_")[0]
                self.imageid_to_path[image_id] = os.path.join(npy_dir, f)         #Pipeline

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        image_id = entry["image_id"]
        english_caption = entry["english_caption"].strip()

        # ✅ Fast lookup
        npy_path = self.imageid_to_path.get(image_id)
        if npy_path is None:
            raise FileNotFoundError(f"No .npy file found for image_id {image_id}")

        feature = np.load(npy_path)
        if len(feature.shape) == 2:
            feature = np.mean(feature, axis=0)

        tokens = self.sp.encode(english_caption, out_type=int)
        decoder_input = [self.bos] + tokens
        decoder_target = tokens + [self.eos]

        decoder_input = decoder_input[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_input))
        decoder_target = decoder_target[:self.max_len] + [self.pad_token] * (self.max_len - len(decoder_target))

        return {
            'image': torch.tensor(feature, dtype=torch.float32),
            'decoder_input': torch.tensor(decoder_input, dtype=torch.long),
            'decoder_target': torch.tensor(decoder_target, dtype=torch.long)
        }


In [ ]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=3, dim_feedforward=2048, dropout=0.1):      #Pipeline
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.image_proj = nn.Linear(2048, d_model)  # ✅ ResNet output size
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask, tgt_padding_mask):
        tgt_emb = self.embedding(tgt)
        tgt_emb = self.pos_encoder(tgt_emb)
        output = self.decoder(tgt=tgt_emb,
                              memory=memory,
                              tgt_mask=tgt_mask,
                              tgt_key_padding_mask=tgt_padding_mask)
        return self.fc_out(output)


In [ ]:
def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones((sz, sz))) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))    #Pipeline
    return mask


In [ ]:
def train_model(model, dataloader, optimizer, criterion, pad_token, num_epochs=10):
    import torch.nn.utils as utils
    from tqdm import tqdm

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}")

        for batch in loop:
            img = batch['image'].to(device)
            dec_in = batch['decoder_input'].to(device)
            dec_out = batch['decoder_target'].to(device)

            tgt = dec_in.transpose(0, 1)  # [T, B]
            tgt_y = dec_out.transpose(0, 1)
            memory = model.image_proj(img).unsqueeze(0)  # [1, B, d_model]

            tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)                  #Training function
            tgt_padding_mask = (dec_in == pad_token).to(device)

            logits = model(tgt, memory, tgt_mask, tgt_padding_mask)
            logits = logits.view(-1, logits.size(-1))
            tgt_y = tgt_y.reshape(-1)

            loss = criterion(logits, tgt_y)
            optimizer.zero_grad()
            loss.backward()
            utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            loop.set_postfix(loss=total_loss / (loop.n + 1e-8))

        print(f"✅ Epoch {epoch+1} | Avg Loss: {total_loss / len(dataloader):.4f}")


In [ ]:
import torch

if torch.cuda.is_available():
    print("✅ CUDA is available")
    print(f"Device count: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("❌ CUDA is NOT available")


✅ CUDA is available
Device count: 1
Current device: 0
Device name: Tesla T4


In [ ]:
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

# === FILE PATHS ===
json_path = "/content/your_dataset_cleaned.json"               # Caption JSON
npy_dir = "/content/ROI_Features_Local"                      # Folder with .npy features
sp_model_path = "/content/english_unigram_4500.model"             # SentencePiece model path

# === DATASET & DATALOADER ===
dataset = LazyImageCaptionDataset(
    json_path=json_path,
    npy_dir=npy_dir,
    sp_model_path=sp_model_path,
    max_len=30,
    pad_token=0
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)                   #Training Block

# === VOCAB SIZE ===
vocab_size = dataset.sp.get_piece_size()

# === MODEL ===
model = TransformerDecoder(vocab_size=vocab_size)

# === LOSS FUNCTION & OPTIMIZER ===
criterion = nn.CrossEntropyLoss(ignore_index=0)  # pad_token=0
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# === TRAIN THE MODEL ===
train_model(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    criterion=criterion,
    pad_token=0,
    num_epochs=30
)


Epoch 1:   0%|          | 0/904 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Epoch 1: 100%|██████████| 904/904 [00:39<00:00, 23.02it/s, loss=4.01]


✅ Epoch 1 | Avg Loss: 4.0013


Epoch 2: 100%|██████████| 904/904 [00:37<00:00, 24.34it/s, loss=3.17]


✅ Epoch 2 | Avg Loss: 3.1632


Epoch 3: 100%|██████████| 904/904 [00:37<00:00, 24.12it/s, loss=2.85]


...

✅ Epoch 28 | Avg Loss: 0.5497


Epoch 29: 100%|██████████| 904/904 [00:37<00:00, 24.09it/s, loss=0.52]


✅ Epoch 29 | Avg Loss: 0.5197


Epoch 30: 100%|██████████| 904/904 [00:37<00:00, 24.22it/s, loss=0.492]

✅ Epoch 30 | Avg Loss: 0.4917


In [ ]:
torch.save(model, "transformer_caption_model.pkl")  # One-liner magic


In [ ]:
model.eval()

TransformerDecoder(
  (embedding): Embedding(4500, 512)
  (pos_encoder): PositionalEncoding()
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inpla

In [ ]:
# Unzip the file
!unzip "/content/drive/MyDrive/Tarika Project/Copy of TestSet_ROI_Features.zip" -d "/content/TestSet_ROI_Features"


Archive:  /content/drive/MyDrive/Tarika Project/Copy of TestSet_ROI_Features.zip
  inflating: /content/TestSet_ROI_Features/2414985_1433.npy  
  inflating: /content/TestSet_ROI_Features/2316154_346.npy  
  inflating: /content/TestSet_ROI_Features/2397030_1271.npy  
  inflating: /content/TestSet_ROI_Features/2397334_1245.npy  
  inflating: /content/TestSet_ROI_Features/2346670_57.npy  
  inflating: /content/TestSet_ROI_Features/2363766_535.npy  
  inflating: /content/TestSet_ROI_Features/2369006_250.npy  
  inflating: /content/TestSet_ROI_Features/333_669.npy  
  inflating: /content/TestSet_ROI_Features/2349729_884.npy  
  inflating: /content/TestSet_ROI_Features/2335350_739.npy  
  inflating: /content/TestSet_ROI_Features/2369726_437.npy  
  inflating: /content/TestSet_ROI_Features/2364293_1119.npy  
  inflating: /content/TestSet_ROI_Features/2322256_583.npy  
  inflating: /content/TestSet_ROI_Features/2322968_1137.npy  
  inflating: /content/TestSet_ROI_Features/2380909_273.npy  
  in

In [ ]:
import torch
import numpy as np

def generate_caption(model, npy_path, sp_model_path, max_len=30, device=None):
    import sentencepiece as spm

    # Load feature
    feature = np.load(npy_path)
    if len(feature.shape) == 2:  # if shape is [N, D], average pool
        feature = np.mean(feature, axis=0)
    feature = torch.tensor(feature, dtype=torch.float32).unsqueeze(0)  # [1, D]

    # Load SentencePiece tokenizer
    sp = spm.SentencePieceProcessor()
    sp.load(sp_model_path)

    bos = sp.bos_id()
    eos = sp.eos_id()
    pad_token = 0
    vocab_size = sp.get_piece_size()

    # Prepare model
    model.eval()
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    feature = feature.to(device)
    memory = model.image_proj(feature).unsqueeze(0)  # [1, 1, d_model]

    # Start decoding loop
    generated = [bos]
    for _ in range(max_len):
        tgt = torch.tensor(generated, dtype=torch.long).unsqueeze(1).to(device)  # [T, 1]
        tgt_mask = generate_square_subsequent_mask(tgt.size(0)).to(device)
        tgt_padding_mask = (tgt.squeeze(1) == pad_token).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(tgt, memory, tgt_mask, tgt_padding_mask)
            next_token_logits = output[-1, 0, :]  # last timestep, batch=0
            next_token = torch.argmax(next_token_logits).item()

        if next_token == eos or next_token == pad_token:
            break
        generated.append(next_token)

    caption = sp.decode(generated[1:])  # Remove BOS before decoding
    return caption


In [ ]:
npy_path = "/content/TestSet_ROI_Features/2317163_1129.npy"
sp_model_path = "/content/english_unigram_4500.model"

caption = generate_caption(model, npy_path, sp_model_path)
print("🖼️ Generated Caption:", caption)


🖼️ Generated Caption: a boy is skating


In [ ]:
import os
import json

# === CONFIG ===
test_dir = "/content/TestSet_ROI_Features"
sp_model_path = "/content/english_unigram_4500.model"
output_json_path = "/content/testset_captions.json"
device = "cuda" if torch.cuda.is_available() else "cpu"

# === Run inference for each .npy file ===
results = []
for fname in sorted(os.listdir(test_dir)):
    if not fname.endswith(".npy"):
        continue
    image_id = fname.replace(".npy", "")  # e.g., "1159518_1165"
    npy_path = os.path.join(test_dir, fname)

    try:
        caption = generate_caption(model, npy_path, sp_model_path, max_len=30, device=device)
        results.append({
            "image_id": image_id,
            "caption": caption
        })
    except Exception as e:
        print(f"⚠️ Error processing {fname}: {e}")

# === Save to JSON ===
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ All captions saved to {output_json_path}")


✅ All captions saved to /content/testset_captions.json


In [ ]:
import json

# === Input paths ===
reference_txt_path = "/content/hindi-visual-genome-test.txt"
inference_json_path = "/content/testset_captions.json"
output_path = "/content/merged_reference_generated.json"

# === Load inference predictions ===
with open(inference_json_path, "r", encoding="utf-8") as f:
    inference_data = json.load(f)

# Build lookup: {base_image_id: generated_caption}
inference_lookup = {}
for item in inference_data:
    full_id = item["image_id"]
    base_id = full_id.split("_")[0]  # "1159288_463" → "1159288"
    inference_lookup[base_id] = item["caption"]

# === Read reference file and merge ===
merged = []
with open(reference_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) < 6:
            continue  # Skip broken lines

        image_id = parts[0]
        english_caption = parts[5].strip()
        generated_caption = inference_lookup.get(image_id)

        if generated_caption is not None:  # Only include matched
            merged.append({
                "image_id": image_id,
                "english_caption": english_caption,
                "generated_caption": generated_caption
            })

# === Save to JSON ===
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print(f"✅ Merged {len(merged)} entries saved to {output_path}")


✅ Merged 1595 entries saved to /content/merged_reference_generated.json
